# Parcellated ISPC — Left-Wing Subjects: Agree vs. Disagree

**Goal**: Test whether neural alignment (ISPC) is higher when left-wing
subjects view content they politically *agree* with versus content they
*disagree* with.

**Condition grouping**:
- **AGREE** = `ProLeft` + `AntiRight` (content that matches left-wing views)
- **DISAGREE** = `AntiLeft` + `ProRight` (content that contradicts left-wing views)

**Contrast**: AGREE > DISAGREE, tested with a split-pool permutation null
distribution (1 000 iterations) and Benjamini–Hochberg FDR correction across
parcels (Schaefer 2018 400-parcel + Tian S3 subcortex).

In [ ]:
from pathlib import Path

# Existing parcellated-ISC API (do not modify this file)
from yy_fmri_kit.event_isc.extraction.parcel import (
    Config,
    load_events,
    load_timeseries,
    extract_post_patterns,
    compute_isc,
    fdr_correct,
    results_to_dataframe,
)

# New contrast utilities
from yy_fmri_kit.event_isc.contrast import (
    filter_subjects_by_group,
    merge_conditions,
    contrast_permutation_test,
)

## 1. Subject Filtering & Configuration

In [ ]:
# ── paths ──────────────────────────────────────────────────────────────────
ROOT            = Path("/path/to/your/project/root")  # <-- EDIT THIS
BEHAVIORAL_CSV  = ROOT / "behavioral_analyses/data/250226/merged_behavioral_bids.csv"
EVENTS_CSV      = ROOT / "behavioral_analyses/data/130426/summary_with_bids_ids.csv"
DATA_DIR        = ROOT / "data/derivatives/parcellated_tian"
OUTPUT_DIR      = ROOT / "data/derivatives/ispc_leftwing_contrast/140426"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── subject selection ──────────────────────────────────────────────────────
# Reads merged_behavioral_bids.csv and keeps only political_group == "left"
left_subs = filter_subjects_by_group(
    BEHAVIORAL_CSV,
    group="left",
    id_col="bids_id",
    group_col="political_group",
)

# ── analysis config ────────────────────────────────────────────────────────
# tsv_glob must match the actual filename pattern under DATA_DIR:
#   sub-N/sub-N_ses-*_task-{run_type}_*atlas-Schaefer2018*timeseries.tsv
cfg = Config(
    data_dir     = DATA_DIR,
    events_csv   = EVENTS_CSV,
    subjects     = left_subs,
    run_types    = ["AntiLeft", "AntiRight", "ProLeft", "ProRight"],
    tr           = 1.0,
    shift_tr     = 4,
    tsv_glob     = "{subject}/{subject}_*_task-{run_type}_*atlas-Schaefer2018*timeseries.tsv",
    subject_col  = "bids_id",
    run_col      = "run",
    post_col     = "post_id",
    onset_col    = "onset_s",
    duration_col = "duration_s",
    n_perms      = 1000,
    fdr_q        = 0.05,
    seed         = 42,
)

print(f"Subjects in analysis: {len(cfg.subjects)}")
print(f"Conditions: {cfg.run_types}")

## 2. Load Events & Timeseries, Extract Post Patterns

In [ ]:
events_df = load_events(cfg)
print(events_df.head())

In [ ]:
# ts_dict: {(subject, run_type): DataFrame(n_trs, n_parcels)}
ts_dict = load_timeseries(cfg)

# parcel names come from the TSV column headers
first_key = next(iter(ts_dict))
parcel_names = ts_dict[first_key].columns.tolist()
print(f"Parcels: {len(parcel_names)}")
print(f"Example parcels: {parcel_names[:5]} ... {parcel_names[-5:]}")

In [ ]:
# patterns: {run_type: {post_id: array(n_subjects, n_parcels)}}
patterns = extract_post_patterns(ts_dict, events_df, cfg)

for rt, posts in patterns.items():
    n_subj = next(iter(posts.values())).shape[0]
    print(f"  {rt}: {len(posts)} posts, {n_subj} subjects")

## 3. Merge Conditions into AGREE / DISAGREE

In [ ]:
CONDITION_MAP = {
    "agree":    ["ProLeft",  "AntiRight"],   # content left-wing subjects agree with
    "disagree": ["AntiLeft", "ProRight"],    # content they disagree with
}

merged = merge_conditions(patterns, CONDITION_MAP)

# sanity: total post count should equal sum across all four conditions
total_merged = sum(len(v) for v in merged.values())
total_orig   = sum(len(v) for v in patterns.values())
assert total_merged == total_orig, (
    f"Post count mismatch after merging: {total_merged} vs {total_orig}"
)
print(f"\nTotal posts (merged): {total_merged} == original {total_orig} ✓")

## 4. Compute ISPC per Condition Group

In [ ]:
isc_agree,    isc_agree_subj    = compute_isc(merged["agree"])
isc_disagree, isc_disagree_subj = compute_isc(merged["disagree"])

print(f"ISPC AGREE    — mean: {isc_agree.mean():.4f},  max: {isc_agree.max():.4f}")
print(f"ISPC DISAGREE — mean: {isc_disagree.mean():.4f},  max: {isc_disagree.max():.4f}")
print(f"\nObserved contrast (agree - disagree): {(isc_agree - isc_disagree).mean():.4f}")

### 4b. ISPC Significance — Phase Randomization

Test whether each condition's ISPC is above chance using phase randomization: \
the activation sequence across posts is randomised in the frequency domain \
per subject (phases shared across all parcels to preserve spatial covariance), \
preserving each subject's autocorrelation structure. \
FDR correction across all 432 parcels (Benjamini–Hochberg, q=0.05).

In [ ]:
from yy_fmri_kit.event_isc.extraction.parcel import permutation_test

# Phase-randomization significance test for AGREE and DISAGREE ISPC
obs_agree,    p_agree,    _ = permutation_test(merged["agree"],    cfg, "isc")
obs_disagree, p_disagree, _ = permutation_test(merged["disagree"], cfg, "isc")

rejected_agree,    p_fdr_agree    = fdr_correct(p_agree,    q=cfg.fdr_q)
rejected_disagree, p_fdr_disagree = fdr_correct(p_disagree, q=cfg.fdr_q)

print(f"AGREE    — significant parcels (FDR q={cfg.fdr_q}): {rejected_agree.sum()} / {len(rejected_agree)}")
if rejected_agree.any():
    for i in rejected_agree.nonzero()[0]:
        print(f"  {parcel_names[i]:50s}  ISC={obs_agree[i]:.4f}  p_fdr={p_fdr_agree[i]:.4f}")

print(f"\nDISAGREE — significant parcels (FDR q={cfg.fdr_q}): {rejected_disagree.sum()} / {len(rejected_disagree)}")
if rejected_disagree.any():
    for i in rejected_disagree.nonzero()[0]:
        print(f"  {parcel_names[i]:50s}  ISC={obs_disagree[i]:.4f}  p_fdr={p_fdr_disagree[i]:.4f}")

# Save
df_agree_sig    = results_to_dataframe(parcel_names, obs_agree,    p_agree,    rejected_agree,    p_fdr_agree)
df_disagree_sig = results_to_dataframe(parcel_names, obs_disagree, p_disagree, rejected_disagree, p_fdr_disagree)
df_agree_sig.to_csv(   OUTPUT_DIR / 'isc_agree_significance.csv',    index=False)
df_disagree_sig.to_csv(OUTPUT_DIR / 'isc_disagree_significance.csv', index=False)
print('Saved isc_agree_significance.csv, isc_disagree_significance.csv')

## 5. AGREE > DISAGREE Contrast — Paired t-test + FDR

In [ ]:
from yy_fmri_kit.event_isc.contrast import ttest_contrast

# Paired t-test per parcel: AGREE > DISAGREE (one-tailed)
t_agree_disagree, p_vals, _, _ = ttest_contrast(
    merged["agree"],
    merged["disagree"],
)

# obs_contrast: raw ISC difference — used for brain maps below
obs_contrast = isc_agree - isc_disagree

rejected, p_fdr = fdr_correct(p_vals, q=cfg.fdr_q)

print(f"Significant parcels (FDR q={cfg.fdr_q}): {rejected.sum()} / {len(rejected)}")
if rejected.any():
    for i in rejected.nonzero()[0]:
        print(f"  {parcel_names[i]:50s}  t={t_agree_disagree[i]:.3f}"
              f"  ISC_diff={obs_contrast[i]:.4f}  p_fdr={p_fdr[i]:.4f}")

## 7. Save Results

In [ ]:
import numpy as np

df = results_to_dataframe(
    parcel_names,
    obs_contrast,
    p_vals,
    rejected,
    p_fdr,
)
df = df.rename(columns={"r": "obs_contrast"})
df["isc_agree"]    = isc_agree
df["isc_disagree"] = isc_disagree

out_csv = OUTPUT_DIR / "agree_vs_disagree_contrast.csv"
df.to_csv(out_csv, index=False)
print(f"Results saved → {out_csv}")
df.head()

## 8. Summary Visualisation

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ── Left: mean ISC per condition group ─────────────────────────────────────
ax = axes[0]
means = [isc_agree.mean(), isc_disagree.mean()]
sems  = [isc_agree.std() / np.sqrt(len(isc_agree)),
         isc_disagree.std() / np.sqrt(len(isc_disagree))]
bars = ax.bar(["Agree", "Disagree"], means, yerr=sems,
              color=["#2196F3", "#F44336"], capsize=5)
ax.set_ylabel("Mean ISPC (r)")
ax.set_title("ISPC by Condition Group\n(mean ± SEM across parcels)")
ax.axhline(0, color="k", lw=0.8, ls="--")

# ── Right: contrast distribution & top parcels ─────────────────────────────
ax = axes[1]
ax.hist(obs_contrast, bins=40, color="#9C27B0", alpha=0.7, label="All parcels")
if rejected.any():
    ax.hist(obs_contrast[rejected], bins=20, color="#FF9800",
            alpha=0.9, label=f"Significant (n={rejected.sum()})")
ax.axvline(0, color="k", lw=0.8, ls="--")
ax.set_xlabel("Observed contrast (agree − disagree ISPC)")
ax.set_ylabel("Parcel count")
ax.set_title("Contrast Distribution\n(FDR-significant parcels highlighted)")
ax.legend()

plt.tight_layout()
fig.savefig(OUTPUT_DIR / "agree_vs_disagree_summary.png", dpi=150)
plt.show()
print(f"Figure saved → {OUTPUT_DIR / 'agree_vs_disagree_summary.png'}")

## 10. Brain Map Helper (local atlas → surface projection)

All brain visualisations below use the local Schaefer+Tian atlas NIfTI to
map parcel values into a 3-D volume and project it onto the fsLR-32k
midthickness surface via `yabplot.project_vol2surf`. This completely avoids
the name-matching issue and covers **all 432 parcels** (400 Schaefer + 32 Tian S3).

In [ ]:
import tempfile
import yabplot as yab
import yabplot.data as ydata
import numpy as np
from yy_fmri_kit.event_isc.contrast import parcels_to_nifti

ATLAS_NII  = ROOT / 'data/atlases/Schaefer2018_tf_2mm_400Parcels7Networks_plus_TianS3.dseg.nii.gz'
LABELS_TSV = ROOT / 'data/atlases/Schaefer2018_400Parcels7Networks_plus_TianS3_labels.tsv'

_lh_surf, _rh_surf = ydata.get_surface_paths('midthickness', 'bmesh')

# ── Cortical helper ────────────────────────────────────────────────────────
def brain_map(
    values, label, *,
    cmap='coolwarm', vminmax=(None, None),
    nan_color=(0.92, 0.92, 0.92),
):
    """
    Map parcel values → NIfTI → fsLR-32k surface projection → yabplot figure.
    Covers all 432 parcels (Schaefer 400 cortical + Tian S3 subcortical).
    Saves PNG to OUTPUT_DIR/{label}.png.
    """
    tmp = Path(tempfile.mktemp(suffix='.nii.gz'))
    parcels_to_nifti(values, parcel_names, ATLAS_NII, LABELS_TSV, tmp)
    lh_data, rh_data = yab.project_vol2surf(str(tmp), interpolation='nearest')
    tmp.unlink(missing_ok=True)
    lh_mesh, rh_mesh = yab.load_vertexwise_mesh(_lh_surf, _rh_surf, lh_data, rh_data)
    views = ['left_lateral', 'left_medial', 'right_lateral', 'right_medial']
    return yab.plot_vertexwise(
        lh_mesh, rh_mesh,
        views=views,
        cmap=cmap,
        vminmax=list(vminmax),
        nan_color=nan_color,
        figsize=(1200, 400),
        display_type='static',
        export_path=str(OUTPUT_DIR / f'{label}.png'),
    )

# ── Subcortical helper ─────────────────────────────────────────────────────
# yabplot ships Tian Scale 1 (16 regions). Our atlas uses Tian S3 (31 parcels,
# finer subdivisions). We average the relevant S3 sub-parcels into each S1
# region before calling plot_subcortical.
S3_TO_S1 = {
    'HIP-rh':  ['pHIP-rh'],                          # only 1 S3 parcel in RH
    'AMY-rh':  ['lAMY-rh',  'mAMY-rh'],
    'pTHA-rh': ['THA-DP-rh','THA-VP-rh'],
    'aTHA-rh': ['THA-VA-rh','THA-DA-rh'],
    'NAc-rh':  ['NAc-shell-rh','NAc-core-rh'],
    'GP-rh':   ['pGP-rh',  'aGP-rh'],
    'PUT-rh':  ['aPUT-rh', 'pPUT-rh'],
    'CAU-rh':  ['aCAU-rh', 'pCAU-rh'],
    'HIP-lh':  ['aHIP-lh', 'pHIP-lh'],
    'AMY-lh':  ['lAMY-lh',  'mAMY-lh'],
    'pTHA-lh': ['THA-DP-lh','THA-VP-lh'],
    'aTHA-lh': ['THA-VA-lh','THA-DA-lh'],
    'NAc-lh':  ['NAc-shell-lh','NAc-core-lh'],
    'GP-lh':   ['pGP-lh',  'aGP-lh'],
    'PUT-lh':  ['aPUT-lh', 'pPUT-lh'],
    'CAU-lh':  ['aCAU-lh', 'pCAU-lh'],
}

def subcortical_map(
    values, label, *,
    cmap='coolwarm', vminmax=(None, None),
    nan_color=(0.92, 0.92, 0.92),
):
    """
    Map parcel values → S1 region means → yabplot plot_subcortical (tian2020_s1).
    Averages our Tian-S3 sub-parcels into the 16 S1 regions that yabplot renders.
    NaN-valued parcels propagate NaN into their S1 region mean.
    Saves PNG to OUTPUT_DIR/{label}_subcortical.png.
    """
    val_dict = {parcel_names[i]: float(values[i]) for i in range(len(values))}
    s1_dict = {}
    for s1_name, s3_names in S3_TO_S1.items():
        sub_vals = [val_dict[n] for n in s3_names if n in val_dict]
        s1_dict[s1_name] = float(np.nanmean(sub_vals)) if sub_vals else float('nan')

    views = ['left_lateral', 'left_medial', 'right_lateral', 'right_medial']
    return yab.plot_subcortical(
        data=s1_dict,
        atlas='tian2020_s1',
        views=views,
        cmap=cmap,
        vminmax=list(vminmax),
        nan_color=nan_color,
        figsize=(1200, 500),
        display_type='static',
        export_path=str(OUTPUT_DIR / f'{label}_subcortical.png'),
    )

# ── Threshold helper ───────────────────────────────────────────────────────
def thresh(values, thr=0.2):
    """Return values with |value| < thr set to NaN (rendered in background colour)."""
    return np.where(np.abs(values) >= thr, values, np.nan)

print('brain_map, subcortical_map, thresh helpers ready.')

## 10b. AGREE / DISAGREE ISPC Maps — Cortical + Subcortical

Uses `brain_map` (vol2surf pipeline, all 432 parcels) and `subcortical_map`
(Tian S3 → S1 aggregation → `plot_subcortical`).

Shared colour scale across AGREE and DISAGREE so they can be compared directly.
Thresholded versions show only parcels where |ISPC| ≥ 0.2.

In [ ]:
# Shared colour scale for AGREE and DISAGREE
isc_scale = [
    float(min(np.nanmin(isc_agree), np.nanmin(isc_disagree))),
    float(max(np.nanmax(isc_agree), np.nanmax(isc_disagree))),
]

# Full arrays (all 432 parcels; subcortical_map extracts indices 400-431)
print('=== AGREE ISPC — cortical ===')
brain_map(isc_agree, 'brain_isc_agree_v2', cmap='Reds', vminmax=isc_scale)

print('\n=== AGREE ISPC — subcortical ===')
subcortical_map(isc_agree, 'brain_isc_agree', cmap='Reds', vminmax=isc_scale)

print('\n=== DISAGREE ISPC — cortical ===')
brain_map(isc_disagree, 'brain_isc_disagree_v2', cmap='Reds', vminmax=isc_scale)

print('\n=== DISAGREE ISPC — subcortical ===')
subcortical_map(isc_disagree, 'brain_isc_disagree', cmap='Reds', vminmax=isc_scale)

In [ ]:
# Thresholded ISPC maps — only parcels with |ISPC| >= 0.2
ISC_THR = 0.2

print(f'=== AGREE ISPC (|ISPC| ≥ {ISC_THR}) — cortical ===')
brain_map(thresh(isc_agree, ISC_THR), 'brain_isc_agree_thr', cmap='Reds', vminmax=(0.01,0.65))

print(f'\n=== AGREE ISPC (|ISPC| ≥ {ISC_THR}) — subcortical ===')
subcortical_map(thresh(isc_agree, ISC_THR), 'brain_isc_agree_thr', cmap='Reds', vminmax=(0.01,0.65))
### 10b-iv. FDR-Significant ISPC Maps — Phase Randomization Null

#Brain maps showing only parcels whose ISPC is significant after FDR correction (Benjamini–Hochberg q=0.05) against the phase-randomization null (Section 4b). Non-significant parcels are rendered in the background colour.
# Significant parcel maps — AGREE and DISAGREE (FDR q=0.05)
# Only FDR-significant parcels are rendered; all others are shown in the
# background colour so the significant regions stand out cleanly.
sig_scale = [
    float(min(np.nanmin(isc_agree), np.nanmin(isc_disagree))),
    float(max(np.nanmax(isc_agree), np.nanmax(isc_disagree))),
]

sig_agree    = np.where(rejected_agree,    obs_agree,    np.nan)
sig_disagree = np.where(rejected_disagree, obs_disagree, np.nan)

print(f"=== AGREE — FDR-significant parcels ({rejected_agree.sum()}) ===")
brain_map(sig_agree,    'isc_agree_sig',    cmap='Reds', vminmax=(0.1,0.7))
subcortical_map(sig_agree,    'isc_agree_sig',    cmap='Reds', vminmax=(0.1,0.7))

print(f"=== DISAGREE — FDR-significant parcels ({rejected_disagree.sum()}) ===")
brain_map(sig_disagree, 'isc_disagree_sig', cmap='Reds', vminmax=(0.1,0.7))
subcortical_map(sig_disagree, 'isc_disagree_sig', cmap='Reds', vminmax=(0.1,0.7))
print(f'\n=== DISAGREE ISPC (|ISPC| ≥ {ISC_THR}) — cortical ===')
brain_map(thresh(isc_disagree, ISC_THR), 'brain_isc_disagree_thr', cmap='Reds', vminmax=isc_scale)

print(f'\n=== DISAGREE ISPC (|ISPC| ≥ {ISC_THR}) — subcortical ===')
subcortical_map(thresh(isc_disagree, ISC_THR), 'brain_isc_disagree_thr', cmap='Reds', vminmax=isc_scale)

In [ ]:
# Agree − Disagree contrast maps — cortical + subcortical, all & thresholded
cmax_agree = float(np.nanmax(np.abs(obs_contrast)))
contrast_vminmax = (-cmax_agree, cmax_agree)
contrast_sig = np.where(rejected, obs_contrast, np.nan)

print('=== CONTRAST (agree − disagree) — cortical ===')
brain_map(obs_contrast, 'brain_contrast_v2', cmap='coolwarm', vminmax=contrast_vminmax)

print('\n=== CONTRAST — subcortical ===')
subcortical_map(obs_contrast, 'brain_contrast', cmap='coolwarm', vminmax=contrast_vminmax)

print('\n=== CONTRAST — FDR-significant only, cortical ===')
brain_map(contrast_sig, 'brain_contrast_sig_v2', cmap='coolwarm', vminmax=contrast_vminmax)

print('\n=== CONTRAST — FDR-significant only, subcortical ===')
subcortical_map(contrast_sig, 'brain_contrast_sig', cmap='coolwarm', vminmax=contrast_vminmax)

print(f'\n=== CONTRAST — |contrast| ≥ {ISC_THR}, cortical ===')
brain_map(thresh(obs_contrast, ISC_THR), 'brain_contrast_thr', cmap='coolwarm', vminmax=contrast_vminmax)

print(f'\n=== CONTRAST — |contrast| ≥ {ISC_THR}, subcortical ===')
subcortical_map(thresh(obs_contrast, ISC_THR), 'brain_contrast_thr', cmap='coolwarm', vminmax=contrast_vminmax)

## 11. Within-AGREE Contrast: AntiRight vs ProLeft

Both conditions contain content that left-wing subjects *agree* with, but
they differ in framing:
- **AntiRight** — outgroup-derogating (criticising the right)
- **ProLeft** — ingroup-affirming (supporting the left)

The contrast tests whether outgroup derogation (`AntiRight`) elicits
stronger neural alignment than ingroup affirmation (`ProLeft`).

**Method**: Paired t-test across the 23 left-wing subjects' per-subject ISC
values, one t-test per parcel; Benjamini–Hochberg FDR correction across parcels.

In [ ]:
# Phase-randomization significance test for AntiRight and ProLeft ISPC
obs_ar, p_ar, _ = permutation_test(patterns["AntiRight"], cfg, "isc")
obs_pl, p_pl, _ = permutation_test(patterns["ProLeft"],   cfg, "isc")

rejected_ar, p_fdr_ar = fdr_correct(p_ar, q=cfg.fdr_q)
rejected_pl, p_fdr_pl = fdr_correct(p_pl, q=cfg.fdr_q)

print(f"AntiRight — significant parcels (FDR q={cfg.fdr_q}): {rejected_ar.sum()} / {len(rejected_ar)}")
if rejected_ar.any():
    for i in rejected_ar.nonzero()[0]:
        print(f"  {parcel_names[i]:50s}  ISC={obs_ar[i]:.4f}  p_fdr={p_fdr_ar[i]:.4f}")

print(f"\nProLeft   — significant parcels (FDR q={cfg.fdr_q}): {rejected_pl.sum()} / {len(rejected_pl)}")
if rejected_pl.any():
    for i in rejected_pl.nonzero()[0]:
        print(f"  {parcel_names[i]:50s}  ISC={obs_pl[i]:.4f}  p_fdr={p_fdr_pl[i]:.4f}")

# Save
df_ar_sig = results_to_dataframe(parcel_names, obs_ar, p_ar, rejected_ar, p_fdr_ar)
df_pl_sig = results_to_dataframe(parcel_names, obs_pl, p_pl, rejected_pl, p_fdr_pl)
df_ar_sig.to_csv(OUTPUT_DIR / 'isc_antiright_significance.csv', index=False)
df_pl_sig.to_csv(OUTPUT_DIR / 'isc_proleft_significance.csv',   index=False)
print('Saved isc_antiright_significance.csv, isc_proleft_significance.csv')

In [ ]:
# Significant parcel maps — AntiRight and ProLeft (FDR q=0.05)
sig_ar = np.where(rejected_ar, obs_ar, np.nan)
sig_pl = np.where(rejected_pl, obs_pl, np.nan)

sig_scale_ar_pl = [
    float(np.nanmin([np.nanmin(sig_ar), np.nanmin(sig_pl)])),
    float(np.nanmax([np.nanmax(sig_ar), np.nanmax(sig_pl)])),
]

print(f"=== AntiRight — FDR-significant parcels ({rejected_ar.sum()}) ===")
brain_map(sig_ar, 'isc_antiright_sig', cmap='coolwarm', vminmax=sig_scale_ar_pl)
subcortical_map(sig_ar, 'isc_antiright_sig', cmap='coolwarm', vminmax=sig_scale_ar_pl)

print(f"=== ProLeft — FDR-significant parcels ({rejected_pl.sum()}) ===")
brain_map(sig_pl, 'isc_proleft_sig', cmap='coolwarm', vminmax=sig_scale_ar_pl)
subcortical_map(sig_pl, 'isc_proleft_sig', cmap='coolwarm', vminmax=sig_scale_ar_pl)

In [ ]:
from yy_fmri_kit.event_isc.contrast import ttest_contrast
from yy_fmri_kit.event_isc.extraction.parcel import fdr_correct, results_to_dataframe

t_ar_pl, p_ar_pl, mu_ar, mu_pl = ttest_contrast(
    patterns['AntiRight'], patterns['ProLeft']
)
rejected_ar_pl, p_fdr_ar_pl = fdr_correct(p_ar_pl, q=cfg.fdr_q)

print(f'Significant parcels (FDR q={cfg.fdr_q}): {rejected_ar_pl.sum()} / {len(rejected_ar_pl)}')
if rejected_ar_pl.any():
    for i in rejected_ar_pl.nonzero()[0]:
        print(f'  {parcel_names[i]:50s}  t={t_ar_pl[i]:.3f}  p_fdr={p_fdr_ar_pl[i]:.4f}')

In [ ]:
# Save results
df_ar_pl = results_to_dataframe(parcel_names, t_ar_pl, p_ar_pl, rejected_ar_pl, p_fdr_ar_pl)
df_ar_pl = df_ar_pl.rename(columns={'r': 't_stat'})
df_ar_pl['isc_anti_right'] = mu_ar
df_ar_pl['isc_pro_left']   = mu_pl
df_ar_pl['isc_diff']       = mu_ar - mu_pl
df_ar_pl.to_csv(OUTPUT_DIR / 'antiright_vs_proleft_ttest.csv', index=False)
print('Saved antiright_vs_proleft_ttest.csv')
df_ar_pl.head()

In [ ]:
# Brain maps — AntiRight vs ProLeft: cortical + subcortical, full & thresholded
isc_scale_ar_pl = [float(min(mu_ar.min(), mu_pl.min())),
                   float(max(mu_ar.max(), mu_pl.max()))]
cmax_ar_pl = float(np.abs(mu_ar - mu_pl).max())

print('AntiRight ISPC — cortical:')
brain_map(mu_ar, 'brain_isc_antiright', cmap='Reds', vminmax=isc_scale_ar_pl)
print('AntiRight ISPC — subcortical:')
subcortical_map(mu_ar, 'brain_isc_antiright', cmap='Reds', vminmax=isc_scale_ar_pl)

print(f'\nAntiRight ISPC (|ISPC| ≥ {ISC_THR}) — cortical:')
brain_map(thresh(mu_ar, ISC_THR), 'brain_isc_antiright_thr', cmap='Reds', vminmax=isc_scale_ar_pl)
print(f'AntiRight ISPC (|ISPC| ≥ {ISC_THR}) — subcortical:')
subcortical_map(thresh(mu_ar, ISC_THR), 'brain_isc_antiright_thr', cmap='Reds', vminmax=isc_scale_ar_pl)

print('\nProLeft ISPC — cortical:')
brain_map(mu_pl, 'brain_isc_proleft', cmap='Reds', vminmax=isc_scale_ar_pl)
print('ProLeft ISPC — subcortical:')
subcortical_map(mu_pl, 'brain_isc_proleft', cmap='Reds', vminmax=isc_scale_ar_pl)

print(f'\nProLeft ISPC (|ISPC| ≥ {ISC_THR}) — cortical:')
brain_map(thresh(mu_pl, ISC_THR), 'brain_isc_proleft_thr', cmap='Reds', vminmax=isc_scale_ar_pl)
print(f'ProLeft ISPC (|ISPC| ≥ {ISC_THR}) — subcortical:')
subcortical_map(thresh(mu_pl, ISC_THR), 'brain_isc_proleft_thr', cmap='Reds', vminmax=isc_scale_ar_pl)

print('\nAntiRight − ProLeft (all parcels) — cortical:')
brain_map(mu_ar - mu_pl, 'brain_antiright_vs_proleft', cmap='coolwarm', vminmax=(-cmax_ar_pl, cmax_ar_pl))
print('AntiRight − ProLeft — subcortical:')
subcortical_map(mu_ar - mu_pl, 'brain_antiright_vs_proleft', cmap='coolwarm', vminmax=(-cmax_ar_pl, cmax_ar_pl))

diff_ar_pl = np.where(rejected_ar_pl, mu_ar - mu_pl, np.nan)
print('\nAntiRight − ProLeft (FDR significant) — cortical:')
brain_map(diff_ar_pl, 'brain_antiright_vs_proleft_sig', cmap='coolwarm', vminmax=(-cmax_ar_pl, cmax_ar_pl))
print('AntiRight − ProLeft (FDR significant) — subcortical:')
subcortical_map(diff_ar_pl, 'brain_antiright_vs_proleft_sig', cmap='coolwarm', vminmax=(-cmax_ar_pl, cmax_ar_pl))

print(f'\nAntiRight − ProLeft (|contrast| ≥ {ISC_THR}) — cortical:')
brain_map(thresh(mu_ar - mu_pl, ISC_THR), 'brain_antiright_vs_proleft_thr', cmap='coolwarm', vminmax=(-cmax_ar_pl, cmax_ar_pl))
print(f'AntiRight − ProLeft (|contrast| ≥ {ISC_THR}) — subcortical:')
subcortical_map(thresh(mu_ar - mu_pl, ISC_THR), 'brain_antiright_vs_proleft_thr', cmap='coolwarm', vminmax=(-cmax_ar_pl, cmax_ar_pl))

## 12. Anti vs Pro Content Contrast

Collapses political affiliation and compares content by **valence**:
- **Anti** = AntiRight + AntiLeft (all outgroup-derogating / attack content)
- **Pro** = ProLeft + ProRight (all ingroup-affirming / supportive content)

Tests whether attack content (regardless of target party) elicits greater
neural alignment than supportive content.

**Method**: Same as Section 11 — paired t-test per parcel, FDR correction.

In [ ]:
# Merge into Anti and Pro groups (also done in the ttest cell below; repeated
# here so the significance test can run independently)
from yy_fmri_kit.event_isc.contrast import merge_conditions

merged_valence = merge_conditions(
    patterns,
    {
        'anti': ['AntiLeft', 'AntiRight'],
        'pro':  ['ProLeft',  'ProRight'],
    }
)

# Phase-randomization significance test for Anti and Pro ISPC
obs_anti, p_anti, _ = permutation_test(merged_valence["anti"], cfg, "isc")
obs_pro,  p_pro,  _ = permutation_test(merged_valence["pro"],  cfg, "isc")

rejected_anti, p_fdr_anti = fdr_correct(p_anti, q=cfg.fdr_q)
rejected_pro,  p_fdr_pro  = fdr_correct(p_pro,  q=cfg.fdr_q)

print(f"Anti — significant parcels (FDR q={cfg.fdr_q}): {rejected_anti.sum()} / {len(rejected_anti)}")
if rejected_anti.any():
    for i in rejected_anti.nonzero()[0]:
        print(f"  {parcel_names[i]:50s}  ISC={obs_anti[i]:.4f}  p_fdr={p_fdr_anti[i]:.4f}")

print(f"\nPro  — significant parcels (FDR q={cfg.fdr_q}): {rejected_pro.sum()} / {len(rejected_pro)}")
if rejected_pro.any():
    for i in rejected_pro.nonzero()[0]:
        print(f"  {parcel_names[i]:50s}  ISC={obs_pro[i]:.4f}  p_fdr={p_fdr_pro[i]:.4f}")

# Save
df_anti_sig = results_to_dataframe(parcel_names, obs_anti, p_anti, rejected_anti, p_fdr_anti)
df_pro_sig  = results_to_dataframe(parcel_names, obs_pro,  p_pro,  rejected_pro,  p_fdr_pro)
df_anti_sig.to_csv(OUTPUT_DIR / 'isc_anti_significance.csv', index=False)
df_pro_sig.to_csv( OUTPUT_DIR / 'isc_pro_significance.csv',  index=False)
print('Saved isc_anti_significance.csv, isc_pro_significance.csv')

In [ ]:
# Significant parcel maps — Anti and Pro (FDR q=0.05)
sig_anti = np.where(rejected_anti, obs_anti, np.nan)
sig_pro  = np.where(rejected_pro,  obs_pro,  np.nan)

sig_scale_ap = [
    float(np.nanmin([np.nanmin(sig_anti), np.nanmin(sig_pro)])),
    float(np.nanmax([np.nanmax(sig_anti), np.nanmax(sig_pro)])),
]

print(f"=== Anti — FDR-significant parcels ({rejected_anti.sum()}) ===")
brain_map(sig_anti, 'isc_anti_sig', cmap='Reds', vminmax=(0.1,0.7))
subcortical_map(sig_anti, 'isc_anti_sig', cmap='Reds', vminmax=(0.1,0.7))

print(f"=== Pro — FDR-significant parcels ({rejected_pro.sum()}) ===")
brain_map(sig_pro, 'isc_pro_sig', cmap='Reds', vminmax=(0.1,0.7))
subcortical_map(sig_pro, 'isc_pro_sig', cmap='Reds', vminmax=(0.1,0.7))

In [ ]:
# Merge into Anti and Pro groups (reuse merge_conditions)
from yy_fmri_kit.event_isc.contrast import merge_conditions

merged_valence = merge_conditions(
    patterns,
    {
        'anti': ['AntiLeft', 'AntiRight'],
        'pro':  ['ProLeft',  'ProRight'],
    }
)

t_anti_pro, p_anti_pro, mu_anti, mu_pro = ttest_contrast(
    merged_valence['anti'], merged_valence['pro']
)
rejected_anti_pro, p_fdr_anti_pro = fdr_correct(p_anti_pro, q=cfg.fdr_q)

print(f'Significant parcels (FDR q={cfg.fdr_q}): {rejected_anti_pro.sum()} / {len(rejected_anti_pro)}')
if rejected_anti_pro.any():
    for i in rejected_anti_pro.nonzero()[0]:
        print(f'  {parcel_names[i]:50s}  t={t_anti_pro[i]:.3f}  p_fdr={p_fdr_anti_pro[i]:.4f}')

In [ ]:
# Save results
df_ap = results_to_dataframe(parcel_names, t_anti_pro, p_anti_pro, rejected_anti_pro, p_fdr_anti_pro)
df_ap = df_ap.rename(columns={'r': 't_stat'})
df_ap['isc_anti'] = mu_anti
df_ap['isc_pro']  = mu_pro
df_ap['isc_diff'] = mu_anti - mu_pro
df_ap.to_csv(OUTPUT_DIR / 'anti_vs_pro_ttest.csv', index=False)
print('Saved anti_vs_pro_ttest.csv')
df_ap.head()

In [ ]:
# Brain maps — Anti vs Pro: cortical + subcortical, full & thresholded
isc_scale_ap = [float(min(mu_anti.min(), mu_pro.min())),
                float(max(mu_anti.max(), mu_pro.max()))]
cmax_ap = float(np.abs(mu_anti - mu_pro).max())

print('Anti ISPC (AntiLeft + AntiRight) — cortical:')
brain_map(mu_anti, 'brain_isc_anti', cmap='Reds', vminmax=isc_scale_ap)
print('Anti ISPC — subcortical:')
subcortical_map(mu_anti, 'brain_isc_anti', cmap='Reds', vminmax=isc_scale_ap)

print(f'\nAnti ISPC (|ISPC| ≥ {ISC_THR}) — cortical:')
brain_map(thresh(mu_anti, ISC_THR), 'brain_isc_anti_thr', cmap='Reds', vminmax=isc_scale_ap)
print(f'Anti ISPC (|ISPC| ≥ {ISC_THR}) — subcortical:')
subcortical_map(thresh(mu_anti, ISC_THR), 'brain_isc_anti_thr', cmap='Reds', vminmax=isc_scale_ap)

print('\nPro ISPC (ProLeft + ProRight) — cortical:')
brain_map(mu_pro, 'brain_isc_pro', cmap='Reds', vminmax=isc_scale_ap)
print('Pro ISPC — subcortical:')
subcortical_map(mu_pro, 'brain_isc_pro', cmap='Reds', vminmax=isc_scale_ap)

print(f'\nPro ISPC (|ISPC| ≥ {ISC_THR}) — cortical:')
brain_map(thresh(mu_pro, ISC_THR), 'brain_isc_pro_thr', cmap='Reds', vminmax=isc_scale_ap)
print(f'Pro ISPC (|ISPC| ≥ {ISC_THR}) — subcortical:')
subcortical_map(thresh(mu_pro, ISC_THR), 'brain_isc_pro_thr', cmap='Reds', vminmax=isc_scale_ap)

print('\nAnti − Pro (all parcels) — cortical:')
brain_map(mu_anti - mu_pro, 'brain_anti_vs_pro', cmap='coolwarm', vminmax=(-cmax_ap, cmax_ap))
print('Anti − Pro — subcortical:')
subcortical_map(mu_anti - mu_pro, 'brain_anti_vs_pro', cmap='coolwarm', vminmax=(-cmax_ap, cmax_ap))

diff_ap = np.where(rejected_anti_pro, mu_anti - mu_pro, np.nan)
print('\nAnti − Pro (FDR significant) — cortical:')
brain_map(diff_ap, 'brain_anti_vs_pro_sig', cmap='coolwarm', vminmax=(-cmax_ap, cmax_ap))
print('Anti − Pro (FDR significant) — subcortical:')
subcortical_map(diff_ap, 'brain_anti_vs_pro_sig', cmap='coolwarm', vminmax=(-cmax_ap, cmax_ap))

print(f'\nAnti − Pro (|contrast| ≥ {ISC_THR}) — cortical:')
brain_map(thresh(mu_anti - mu_pro, ISC_THR), 'brain_anti_vs_pro_thr', cmap='coolwarm', vminmax=(-cmax_ap, cmax_ap))
print(f'Anti − Pro (|contrast| ≥ {ISC_THR}) — subcortical:')
subcortical_map(thresh(mu_anti - mu_pro, ISC_THR), 'brain_anti_vs_pro_thr', cmap='coolwarm', vminmax=(-cmax_ap, cmax_ap))